In [3]:
import os
import pandas as pd
import platform
import sys
import torch
import pickle

from transformers import GPT2Tokenizer, AutoTokenizer
import sqlite3
import tqdm
from torch.nn import functional as F

from string import punctuation
from nltk import pos_tag

import re 

In [4]:

# Things to do
# 1) Load Model and tokenizer

# 2) Load Dataset
#   2.1) 2 different datasets - Natural Stories and Dundee Corpus

# 3) Tokenize the dataset
#   3.1) Preprocess the dataset(?????)

# 4) Function to calculate surprisal given model and sentence


# 5) Store in sqlite database????
#  5.1) Table structure 


#Question to answer
# What is the most efficient way to sync up data between remote server and local machine
# 1) Option A- Use Excel in both places
# 2) Option B - Use sqlite database
#  2.1) Use SQLLite database in local machine and have a csv file in remote server and write a script to download the csv file and update the sqlite database (Probably the best option?)




In [5]:
if "pop-os" in platform.node():
    ROOT = r"/home/abishekthamma/PycharmProjects/masters_thesis/ss-llm/nanoGPT/"
else:
    ROOT = r'/gpfs/home4/athamma/repo/ss-llm/nanoGPT/'
    
TOKENIZER_ROOT = os.path.join(ROOT, "data")
OUT_ROOT = os.path.join(ROOT, "output_dump")
RESULTS_ROOT = os.path.join(ROOT, "results")

SQL_DB = os.path.join(RESULTS_ROOT, "results.db")



In [ ]:
def create_connection_cursor(db_file):
    """
    Create a database connection to the SQLite database specified by the db_file

    Args:
        db_file (str): database file

    Returns:
        Connection object or None
    """
    conn = sqlite3.connect(db_file)
    c = conn.cursor()
    return conn, c

conn, c = create_connection_cursor(SQL_DB)


In [13]:
def load_model(model_id, device="cuda"):
    """
    Loads a pre-trained GPT model from a checkpoint file.

    Args:
        out_dir (str): The directory where the checkpoint file is located.
        device (torch.device): The device to load the model onto.

    Returns:
        GPT: The loaded GPT model.

    Raises:
        FileNotFoundError: If the checkpoint file is not found.
    """
    conn, c = create_connection_cursor(SQL_DB)
    c.execute("SELECT OutputFolderName FROM Model WHERE ModelID=?", (model_id,))
    out_dir = c.fetchone()[0]
    conn.close()
    
    out_dir = os.path.join(OUT_ROOT, out_dir)
    ckpt_path = os.path.join(out_dir, 'ckpt.pt')
    print(f"Loading model from {ckpt_path}")
    # NANOGPT_ROOT = str(Path(__file__).parents[4])

    # Add if condition to check if inside server and if is, then add the path correctly. Default is local for now
    if "pop-os" in platform.node():
        NANOGPT_ROOT = r'/home/abishekthamma/PycharmProjects/masters_thesis/ss-llm/nanoGPT'  # Edit later to be dynamic
    else:
        NANOGPT_ROOT = r'/gpfs/home4/athamma/repo/ss-llm/nanoGPT'
    sys.path.append(NANOGPT_ROOT)
    from model import GPT, GPTConfig

    checkpoint = torch.load(ckpt_path, map_location=device)

    # Backward compatibility for new model args for QKV and FFW Adjustments
    if checkpoint["model_args"].get("wm_decay_length", None) is None:
        # wm_decay_length = block_size
        checkpoint["model_args"]["wm_decay_length"] = checkpoint["model_args"]["block_size"]
    # Setting head size as 3 times n_embd if not set already
    if checkpoint['model_args'].get('head_size_qkv', None) is None:
        checkpoint['model_args']['head_size_qkv'] = checkpoint['model_args']['n_embd']

    if checkpoint["model_args"].get("ffw_dim", None) is None:
        checkpoint["model_args"]["ffw_dim"] = 4 * checkpoint["model_args"]["n_embd"]

    # print(checkpoint['model_args'])
    gptconf = GPTConfig(**checkpoint['model_args'])

    load_model = GPT(gptconf)

    state_dict = checkpoint['model']
    unwanted_prefix = '_orig_mod.'
    for k, v in list(state_dict.items()):
        if k.startswith(unwanted_prefix):
            state_dict[k[len(unwanted_prefix):]] = state_dict.pop(k)

    load_model.load_state_dict(state_dict)
    load_model.eval()

    load_model = load_model.to(device)

    return load_model

def load_tokenizer(data_dir):
    """
    Load tokenizer for natural stories evaluation.

    Args:
        data_dir (str): The directory path where the tokenizer data is stored.

    Returns:
        tokenizer (Tokenizer): The loaded tokenizer object.

    Raises:
        NotImplementedError: If stoi/itos is not supported or found.

    """
    meta_path = os.path.join(data_dir, 'meta.pkl')
    load_meta = os.path.exists(meta_path)
    if load_meta:
        with open(meta_path, 'rb') as f:
            meta = pickle.load(f)
        if meta.get("custom_tokenizer", False):
            print(f"Loading custom tokenizer from {data_dir}")
            tokenizer = AutoTokenizer.from_pretrained(data_dir, use_fast=False)
        else:
            if meta.get("stoi", False):
                raise NotImplementedError("stoi/itos not supported yet")
            else:
                raise NotImplementedError("No stoi/itos found")
    else:
        print("No meta.pkl found, using default GPT-2 tokenizer")
        tokenizer = GPT2Tokenizer.from_pretrained("openai-community/gpt2")

    if not tokenizer.eos_token:
        tokenizer.add_special_tokens({"eos_token": "</s>"})
    if not tokenizer.pad_token:
        tokenizer.pad_token = tokenizer.eos_token

    tokenizer.padding_side = "left"  # Add if needed?
    return tokenizer

def load_model_tokenizer(out_dir, data_dir, device="cuda"):
    model = load_model(out_dir, device)
    tokenizer = load_tokenizer(data_dir)
    return model, tokenizer

def return_surprisals(model, context_window_tensor, output_tensor, device='cuda'):
    """
    Given a model. given a context window, give a token, return the surprisal score for the token 
    :param model: 
    :param context_window: (batch_size, context_window) (A list of tokens, needs to be converted to tensor and could be unequal length, so pad with 0 to the left. 0 is "<|endoftext|>" token)
    :param token_list: (batch_size, 1)
    :param device: 
    :return: 
    """
    
    model = model.to(device)
    context_window_tensor = context_window_tensor.to(device)
    output_tensor = torch.tensor(output_tensor).to(device)
    
    #token_tensor = torch.tensor(token_list).unsqueeze(0).to(device)
    with torch.no_grad():
        logits, _ = model(context_window_tensor) #probably don't need the second tensor
    probs = F.log_softmax(logits, dim=-1)
    #print(probs.shape)
    #probs has shape (batch_size, 1, vocab_size)
    #
    #Use the output tensor to get the log probability of the token
    
    token_logprob =  probs.gather(2, output_tensor.unsqueeze(1)).squeeze()
    #print("H20", probs.gather(2, output_tensor.unsqueeze(1)).shape)
    #SANITY CHECK
    #print("H21", token_logprob.shape, token_logprob.squeeze().shape)
    #print("H22", probs[0, 0, output_tensor[0]])
    return -token_logprob



In [14]:
# Things to want to store in the database

# Natural Stories Corpus - 10 Stories (Item is column name) - Each store has a zone (Which is each word) 
# Dundee Corpus (Read and fill) 

# Depending on the tokenizer used - Each word could be tokenized into one or more words differently
# (Is there any use for storing this?) - IDK

# Model returns surprisal scores for each token, however, depending in some tokens this could be multiple tokens, in which case we need to merge them (Check how we did it, I think  we did sum(?))

# For each model, we have surprisal score for each word (Sentence level(?) or Entire Story level)


# Natural Stories Corpus has reading time for each word (>1 participant) for each story


# Model Table - Store Model ID, Tokenizer ID, (And other model specific details add later if needed) 

# Tokenizer Table - Store Tokenizer ID, Tokenizer Name, Vocab Size, (And other tokenizer specific details add later if needed)

# Story Table - Corpus ID, Story ID, Word ID, (Other details that might be needed for later analysis - Such as number of characters of the word, position of word in sentence (?), position of word in story (?), Sentence ID (?) )

# Separate table to store how a tokenizer tokenized a certain story (?) 
  # YES
  # Since each tokenizer tokenizes differently, we need to store how it tokenized each word in a story
  # Each story, each word would have 1 or more tokens
  # Tokenizer ID, Story ID, Word ID, Token Key (start at 1 and go to how many ever needed, multiple rows for each word), Token Value (The actual token)
  # (Or is it better 
  # What is the use of this? Can query it story wise and get list of tokens to pass into the model and not need to tokenize again
  

# Dependent variable tables  - 
    # In case of Natu53ral Stories Corpus - Reading Time per word, per story, per participant
    # In case of Dundee Corpus - (Read and fill)





In [15]:
# Model Table -
# For now - Columns are ModelID, OutputFolderName, TokenizerID

#Tokenizer table
# For now - Columns are TokenizerID, TokenizerName, VocabularySize

# Story Table - CorpusID, StoryID, WordID, Word, (Other things could include - Number of characters in the word, Position of word in sentence, Position of word in story, SentenceID)

# TokenizedStory Table - TSID, TokenizerID, StoryID, WordID, TokenKey, TokenValue

# SurprisalScore Table - ModelID, StoryID, WordID, SurprisalScore

In [16]:
# Copy and write more accurate ones after edits by copying from the database itself 


# #CREATE TOKENIZER TABLE
# c.execute('''CREATE TABLE IF NOT EXISTS Tokenizer (
#             TokenizerID INTEGER NOT NULL PRIMARY KEY, 
#             TokenizerName TEXT NOT NULL, 
#             VocabularySize INTEGER NOT NULL)''')

# #CREATE MODEL TABLE
# c.execute('''CREATE TABLE IF NOT EXISTS Model (
#             ModelID INTEGER NOT NULL UNIQUE, 
#             OutputFolderName TEXT NOT NULL, 
#             TokenizerID INTEGER NOT NULL,
#             NumLayers INTEGER NOT NULL,
#             NumHeads INTEGER NOT NULL,
#             BlockSize INTEGER NOT NULL,
#             EmbeddingDimension INTEGER NOT NULL,
#             BatchSize INTEGER NOT NULL,
#             LearningRate REAL NOT NULL,
#             Seed INTEGER NOT NULL,
#             Masking BOOLEAN NOT NULL,
#             MaskType TEXT NOT NULL,
#             MaskDecayRate REAL NOT NULL,
#             EchoicMemory INTEGER NOT NULL,
#             CurriculumLearning BOOLEAN NOT NULL,
#             CurriculumType TEXT,
#             Dataset TEXT NOT NULL,
#             FOREIGN KEY(TokenizerID) REFERENCES Tokenizer(TokenizerID))
#             ''')
# #Add a column for Iterations later


# #CREATE STORY TABLE
# c.execute('''CREATE TABLE IF NOT EXISTS Story (
#             StoryWordID INTEGER NOT NULL PRIMARY KEY,
#             CorpusID INTEGER NOT NULL,
#             StoryID INTEGER NOT NULL,
#             WordID INTEGER NOT NULL,
#             Word TEXT NOT NULL,
#             CONSTRAINT unique_story_word UNIQUE (CorpusID, StoryID, WordID))  
#             ''')
# #On CONFLICT ???

# #CREATE TOKENIZED STORY TABLE
# c.execute('''CREATE TABLE IF NOT EXISTS TokenizedStory (
# 	        TSID    INTEGER NOT NULL UNIQUE,
#             TokenizerID INTEGER NOT NULL,
#             StoryWordID INTEGER NOT NULL,
#             TokenKey INTEGER NOT NULL,
#             TokenValue TEXT NOT NULL,
#             PRIMARY KEY(TSID AUTOINCREMENT),
#             FOREIGN KEY(StoryWordID) REFERENCES Story(StoryWordID))
#             ''')

# #Create table for word UID (Each unique word, without any capitalization or punctuation or special characters)
# #Table contains columns WordUID, Word, character length of the word (And other word level details that we might need later, Unigram frequency(?), Word Class (?))

# c.execute('''CREATE TABLE IF NOT EXISTS WordDetails (
#             WordUID INTEGER NOT NULL PRIMARY KEY,
#             Word TEXT NOT NULL,
#             CharacterLength INTEGER NOT NULL)''')


# #CREATE TABLE TO STORE READING TIME FOR NATURAL STORIES CORPUS 
# #Table structure - RTUID, WorkerID, RT, CorrectAnswers,  StoryWordID (Foreign Key to Story Table)

# c.execute('''CREATE TABLE IF NOT EXISTS SPRTNaturalStories (
#             RTUID INTEGER NOT NULL PRIMARY KEY,
#             WorkerID TEXT NOT NULL,
#             RT REAL NOT NULL,
#             CorrectAnswers INTEGER NOT NULL,
#             StoryWordID INTEGER NOT NULL,
#             FOREIGN KEY(StoryWordID) REFERENCES Story(StoryWordID))''')

# #Add constraint that WorkerID and StoryWordID are unique
# #c.execute('''CREATE UNIQUE INDEX IF NOT EXISTS unique_worker_story ON SPRTNaturalStories (WorkerID, StoryWordID)''')

# #CREATE SURPRISAL SCORE TABLE
# #Fo

In [8]:
#Read the Natural Stories Corpus

natural_stories_path = "/home/abishekthamma/PycharmProjects/masters_thesis/ss-llm/nanoGPT/eval/natural_stories/naturalstories_RTS/all_stories.tok"
natural_stories_df = pd.read_csv(natural_stories_path, sep="\t")
natural_stories_df.rename(columns={"word": "Word", "zone": "WordID", "item": "StoryID"}, inplace=True)
natural_stories_df["CorpusID"] = 1

natural_stories_df.head()

#Insert into the Story Table
natural_stories_df.to_sql("Story", conn, if_exists="append", index=False)


,Word,WordID,StoryID,CorpusID
0,If,1,1,1
1,you,2,1,1
2,were,3,1,1
3,to,4,1,1
4,journey,5,1,1


In [26]:

#Read Story table and for each word, add to the WordDetails table if not already present, and get the WordUID and add to the Story table

c.execute("SELECT * FROM Story WHERE CorpusID=1")


def clean_word(word_str):
    """
    
    """
    #clean the word by removing punctuation, capitalization, special characters (Keep hyphen and apostrophe)
    custom_punctuation = punctuation.replace("'", "").replace("-", "")    
    word_str = word_str.lower()
    word_str = word_str.translate(str.maketrans('', '', custom_punctuation))
    
    if word_str[0] == "'":
        word_str = word_str[1:]
    
    if word_str[-1] == "'":
        word_str = word_str[:-1]

    return word_str
    

for row in tqdm.tqdm(c.fetchall()):
    storywordid, corpusid, storyid, wordid, word, worduid = row

    clean_word_str = clean_word(word)

    c.execute("SELECT WordUID FROM WordDetails WHERE Word=?", (clean_word_str,))
    if not c.fetchone():
        c.execute("INSERT INTO WordDetails (Word, CharacterLength) VALUES (?, ?)", (clean_word_str, len(clean_word_str)))
        conn.commit()
        c.execute("SELECT WordUID FROM WordDetails WHERE Word=?", (clean_word_str,))
        worduid = c.fetchone()[0]
        c.execute("UPDATE Story SET WordUID=? WHERE StoryWordID=?", (worduid, storywordid))
        conn.commit()
    
    else:
        c.execute("SELECT WordUID FROM WordDetails WHERE Word=?", (clean_word_str,))
        worduid = c.fetchone()[0]
        c.execute("UPDATE Story SET WordUID=? WHERE StoryWordID=?", (worduid, storywordid))
        conn.commit()



100%|██████████| 10256/10256 [00:47<00:00, 216.41it/s]


In [16]:
from sqlite3 import IntegrityError

#Dump Self Paced Reading Time results from the Natural Stories Corpus into the database
reading_time_path = r'/home/abishekthamma/PycharmProjects/masters_thesis/ss-llm/nanoGPT/eval/natural_stories/naturalstories_RTS/processed_RTs.tsv'
processed_RTs_df = pd.read_csv(reading_time_path, sep="\t")
processed_RTs_df = processed_RTs_df.rename(columns={
    "WorkerId": "WorkerID",
    "item":"StoryID",
    "zone":"WordID",
    "nItem":"ItemSubjectCount",
    "word": "Word",
    "correct": "CorrectAnswers",
})
processed_RTs_df["CorpusID"] = 1
processed_RTs_df.sort_values(["WorkerID", "StoryID", "WordID"])


#Separate Summary statistics and non summary statistics into separate dataframes
processed_RTs_summary_df = processed_RTs_df[["CorpusID","StoryID", "WordID", "Word", "ItemSubjectCount", "meanItemRT", "sdItemRT", "gmeanItemRT", "gsdItemRT"]]
processed_RTs_subject_df = processed_RTs_df[["CorpusID", "StoryID", "WordID", "Word", "WorkerID", "RT", "CorrectAnswers"]]

#Read the story table from the database and get the unique id (Row id?) for 
# CorpusID, StoryID, WordID and then use that as a foreign key for the new tables

story_df = pd.read_sql("SELECT * FROM Story WHERE CorpusID=1", conn)

processed_RTs_subject_df = processed_RTs_subject_df.merge(story_df[["CorpusID", "StoryID", "WordID", "Word", "StoryWordID"]] 
                                                          , on=["CorpusID", "StoryID", "WordID", "Word"], how="left")
processed_RTs_subject_df = processed_RTs_subject_df[["StoryWordID","WorkerID","RT", "CorrectAnswers"]]
processed_RTs_subject_df.head(10)

#Insert into the database
try:
    processed_RTs_subject_df.to_sql("SPRTNaturalStories", conn, if_exists="append", index=False)
except IntegrityError as e:
    print("Integrity Error, Data already exists in the database")
    print(e)

#Do we need to store summary statistics or is it enough to store the individual subject data? 
#Can easily derive summary statistics from the individual subject data if needed?

Integrity Error, Data already exists in the database
UNIQUE constraint failed: SPRTNaturalStories.WorkerID, SPRTNaturalStories.StoryWordID


In [48]:
story_df = pd.read_sql("""SELECT CorpusID, StoryID, group_concat(StoryWordID) as StoryWordIDList, group_concat(WordDetails.Word, " ") AS Story
from Story
Join WordDetails on Story.WordUID = WordDetails.WordUID
GROUP BY CorpusID, StoryID""", conn)

#run nltk.download('universal_tagset') if not already downloaded
#Using Universal tagset for POS tagging for now (For Simplicity?)
story_df["tagged_story"] = story_df["Story"].apply(lambda x: pos_tag(x.split(" "), tagset="universal"))
story_df["StoryWordIDList"] = story_df["StoryWordIDList"].apply(lambda x: x.split(","))
story_df["Story"] = story_df["Story"].apply(lambda x: x.split(" "))

assert all(story_df["Story"].apply(len) == story_df["StoryWordIDList"].apply(len))
assert all(story_df["Story"].apply(len) == story_df["tagged_story"].apply(len))

#print(story_df.head())
story_df = story_df.set_index(["CorpusID", "StoryID"]).apply(pd.Series.explode).reset_index()
story_df["tagged_story"] = story_df["tagged_story"].apply(lambda x: x[1])
story_df = story_df.rename(columns={"StoryWordIDList": "StoryWordID", "tagged_story": "POS_Tag"})
story_df["StoryWordID"] = story_df["StoryWordID"].astype(int)

story_df.head()




,CorpusID,StoryID,StoryWordID,Story,POS_Tag
0,1,1,1,if,ADP
1,1,1,2,you,PRON
2,1,1,3,were,VERB
3,1,1,4,to,PRT
4,1,1,5,journey,VERB


In [49]:
#Use StoryWordID to insert back into Story table

c.executemany("UPDATE Story SET POSTag=? WHERE StoryWordID=?", story_df[["POS_Tag", "StoryWordID"]].values)
conn.commit()

In [42]:
# Add wordfrequency to the WordDetails table

subtlex_frequencies_path = "SUBTLEXusExcel2007.xlsx"
subtlex_frequencies_df = pd.read_excel(subtlex_frequencies_path, dtype={"Word": str}, keep_default_na=False)

#Make word lowercase
subtlex_frequencies_df["Word"] = subtlex_frequencies_df["Word"].str.lower()
subtlex_frequencies_df["Word"] = subtlex_frequencies_df["Word"].astype(str)

assert subtlex_frequencies_df[subtlex_frequencies_df.Word.duplicated()].shape[0] == 0
assert subtlex_frequencies_df[subtlex_frequencies_df.Word.map(lambda x: any(char in punctuation for char in x))].shape[0] == 0
subtlex_frequencies_dict = subtlex_frequencies_df[['Word', 'Lg10WF']].set_index("Word").to_dict(orient="dict")["Lg10WF"]

subtlex_frequencies_df

# #Check for words with punctuation
# subtlex_frequencies_df[subtlex_frequencies_df["Word"].str.contains("[^a-zA-Z0-9\s]")]

,Word,FREQcount,CDcount,FREQlow,Cdlow,SUBTLWF,Lg10WF,SUBTLCD,Lg10CD
0,the,1501908,8388,1339811,8388,29449.176471,6.176644,100.000000,3.923710
1,to,1156570,8383,1138435,8380,22677.843137,6.063172,99.940391,3.923451
2,a,1041179,8382,976941,8380,20415.274510,6.017526,99.928469,3.923399
3,you,2134713,8381,1595028,8376,41857.117647,6.329340,99.916547,3.923348
4,and,682780,8379,515365,8374,13387.843137,5.834281,99.892704,3.923244
...,...,...,...,...,...,...,...,...,...
74281,zoroastrian,1,1,0,0,0.019608,0.301030,0.011922,0.301030
74282,zoroastrianism,1,1,0,0,0.019608,0.301030,0.011922,0.301030
74283,zugzwang,1,1,1,1,0.019608,0.301030,0.011922,0.301030
74284,zygotes,1,1,1,1,0.019608,0.301030,0.011922,0.301030


In [75]:
worddetails_df = pd.read_sql("SELECT * FROM WordDetails", conn)
worddetails_df["Frequencies"] = worddetails_df["Word"].apply(lambda x: subtlex_frequencies_dict.get(x, 0))
print("Words with no frequency data = ", worddetails_df[worddetails_df["Frequencies"] == 0].shape[0])
worddetails_df

Words with no frequency data =  117


,WordUID,Word,CharacterLength,Frequencies
0,1,if,2,5.256744
1,2,you,3,6.329340
2,3,were,4,4.928421
3,4,to,2,6.063172
4,5,journey,7,3.007748
...,...,...,...,...
2365,2366,hope,4,4.213597
2366,2367,leads,5,3.004321
2367,2368,diagnostic,10,1.863323
2368,2369,tools,5,2.754348


In [77]:
#Update the WordDetails table with the frequency data

c.executemany("UPDATE WordDetails SET LogFrequencies=? WHERE WordUID=?", worddetails_df[["Frequencies", "WordUID"]].values)
conn.commit()

In [1]:
#Dump surprisal scores for models from previously analysed stuff from excel into the database. 
import pandas as pd

story_surprisal_keys_csv_df = pd.read_csv("/home/abishekthamma/PycharmProjects/masters_thesis/ss-llm/nanoGPT/eval/natural_stories/story_surprisal_keys.csv")
story_surprisal_values_csv_df = pd.read_csv("/home/abishekthamma/PycharmProjects/masters_thesis/ss-llm/nanoGPT/eval/natural_stories/storyword_model_surprisals.csv")

print(story_surprisal_keys_csv_df.columns)
print(story_surprisal_values_csv_df.columns)

combined_story_surprisal_csv_df = story_surprisal_values_csv_df.merge(story_surprisal_keys_csv_df, on=["storyword_UID"])
combined_story_surprisal_csv_df.head()

Index(['storyword_UID', 'item', 'zone', 'word', 'tokens', 'tokenizer'], dtype='object')
Index(['model_id', 'storyword_UID', 'surprisal'], dtype='object')


,model_id,storyword_UID,surprisal,item,zone,word,tokens,tokenizer
0,5445338,20512.0,13.324539,1,1,If,[331],babylm_wocdes_full_bpe
1,5445338,20513.0,3.733041,1,2,you,[210],babylm_wocdes_full_bpe
2,5445338,20514.0,3.349813,1,3,were,[390],babylm_wocdes_full_bpe
3,5445338,20515.0,2.085532,1,4,to,[196],babylm_wocdes_full_bpe
4,5445338,20516.0,10.150342,1,5,journey,[4751],babylm_wocdes_full_bpe


In [7]:
#Using item zone amd use it to match it 

conn, c = create_connection_cursor(SQL_DB)
story_df = pd.read_sql(""" SELECT StoryWordID, StoryID, WordID FROM Story WHERE CorpusID=1""", conn)
combined_story_surprisal_csv_df = combined_story_surprisal_csv_df.merge(story_df, left_on=["item", "zone"], right_on=["StoryID", "WordID"])

combined_story_surprisal_csv_df.head()


,model_id,storyword_UID,surprisal,item,zone,word,tokens,tokenizer,StoryWordID,StoryID,WordID
0,5445338,20512.0,13.324539,1,1,If,[331],babylm_wocdes_full_bpe,1,1,1
1,5445338,20513.0,3.733041,1,2,you,[210],babylm_wocdes_full_bpe,2,1,2
2,5445338,20514.0,3.349813,1,3,were,[390],babylm_wocdes_full_bpe,3,1,3
3,5445338,20515.0,2.085532,1,4,to,[196],babylm_wocdes_full_bpe,4,1,4
4,5445338,20516.0,10.150342,1,5,journey,[4751],babylm_wocdes_full_bpe,5,1,5


In [8]:
combined_story_surprisal_csv_df["model_id"].unique()

array([5445338, 5492134, 5496426, 5983308, 6689753, 5444724, 5492054,
       5496427, 5734459, 5734464, 5734465, 5734467, 5734550, 5757736,
       5757737, 5983309, 5988018, 5988019, 5988020, 5988022, 5989080,
       5989082, 6486043, 6486044, 6603578, 6603579, 6603580, 6607670,
       6620547, 6620548, 6621801, 6681922, 6681937, 6681938, 6681939,
       6681940, 6681941, 6681942, 6681944, 6681945, 6681946, 6681947,
       6681948, 6681949, 6681950, 6681951, 6683308, 6683309, 6683310,
       6683311, 6689751, 6689752, 6810203, 6810205, 6810296, 6810297,
       6810320, 6810321, 6810323, 6810325, 6810326, 6839399, 6839402,
       6839403, 6839404, 6839405, 6839409, 6839410, 6839411, 6839412,
       6839413, 6839414, 6839416, 6839417, 6839418, 6839419, 6839420,
       6839421, 6839422, 6839423, 6839424, 6839425, 6839426, 6839427,
       6839428, 6839429, 6839430, 6839431, 6849723, 6849725, 6864685,
       6890225, 6890228, 6890229, 6890230, 6890231, 6890232, 6890233,
       6890234, 6890

In [9]:
# Create surprisal scores table  
# Use only necessary columns - ModelID, StoryWordID, SurprisalScore

# c.execute('''CREATE TABLE IF NOT EXISTS ModelSurprisalScores (
#             ModelID INTEGER NOT NULL,
#             StoryWordID INTEGER NOT NULL,
#             SurprisalScore REAL NOT NULL,
#             PRIMARY KEY(ModelID, StoryWordID),
#             FOREIGN KEY(ModelID) REFERENCES Model(ModelID),
#             FOREIGN KEY(StoryWordID) REFERENCES Story(StoryWordID))''')


# ModelID will match with model table, StoryWordID will match with Story table

# Since code already sums up surprisal scores across multiple tokens for a word, we don't need to do any joins at a token level

#Insert into the database

#But before that check if data is clean and ready to be inserted

# Count different StoryWordIDs in the combined_story_surprisal_csv_df and the Story table
print("Count different StoryWordIDs in the combined_story_surprisal_csv_df and the Story table ", combined_story_surprisal_csv_df["StoryWordID"].nunique(), story_df["StoryWordID"].nunique())

print("Number of models being inserted ", combined_story_surprisal_csv_df["model_id"].nunique())

# Check if all models are present in the Model table
print("Number of models present in the Model table ", pd.read_sql("SELECT ModelID FROM Model", conn)["ModelID"].nunique())
print(set(combined_story_surprisal_csv_df["model_id"].unique().tolist()) - set(pd.read_sql("SELECT ModelID FROM Model", conn)["ModelID"].unique().tolist()))


print(combined_story_surprisal_csv_df.shape[0], combined_story_surprisal_csv_df["StoryWordID"].nunique()*combined_story_surprisal_csv_df["model_id"].nunique())

#Filter only unique models that are not already present in the Model Surprisal Scores table

combined_story_surprisal_csv_df = combined_story_surprisal_csv_df[~combined_story_surprisal_csv_df["model_id"].isin(pd.read_sql("SELECT ModelID FROM ModelSurprisalScores", conn)["ModelID"].unique())]

print("Number of models being inserted ", combined_story_surprisal_csv_df["model_id"].nunique())

#Insert into the database

c.executemany("INSERT INTO ModelSurprisalScores (ModelID, StoryWordID, SurprisalScore) VALUES (?, ?, ?)", 
                combined_story_surprisal_csv_df[["model_id", "StoryWordID", "surprisal"]].values)

conn.commit()


Count different StoryWordIDs in the combined_story_surprisal_csv_df and the Story table  10256 10256
Number of models being inserted  255
Number of models present in the Model table  190
{8111938, 8178694, 8178952, 8111939, 8111940, 8111941, 8111942, 8178972, 8117279, 8117280, 8117281, 8117282, 8117283, 8180897, 8180899, 8465733, 8098216, 8111923, 8111924, 8465077, 8111925, 6607670, 8111928, 8173238, 8465082, 8173239, 8465084, 8465085, 8465086, 8465087, 8173240, 8465089, 8465090, 8465091, 8465604, 8465093, 8465605, 8117319, 8465607, 8117321, 8117322, 8465610, 8465611, 8117325, 8117326, 8465612, 8465608, 8456913, 8465609, 8456915, 8058703, 8058704, 8173522, 8173525, 8182743, 8182744, 8182745, 8464992, 8173537, 8175589, 8173546, 8171121, 8170355, 8170356, 8171125, 8096895}
2615280 2615280
Number of models being inserted  22


In [8]:
#Reading time fit results plotting
import pandas as pd
reading_time_path = "/home/abishekthamma/PycharmProjects/masters_thesis/ss-llm/nanoGPT/eval/reading_time_analysis/surprisal_analysis_results_temp.csv"
reading_time_lmer = pd.read_csv(reading_time_path)



reading_time_lmer = reading_time_lmer[["ModelID", "Log-Likelihood", "Coefficient for Surprisal Score", "Delta Log-Likelihood", "AIC", "BIC"]]
reading_time_lmer




,ModelID,Log-Likelihood,Coefficient for Surprisal Score,Delta Log-Likelihood,AIC,BIC
0,5734459,-169650.372658,0.025981,1721.849012,339314.745315,339396.306095
1,5734464,-169644.225553,0.025955,1727.996117,339302.451107,339384.011887
2,5444724,-169708.751421,0.026064,1663.470249,339431.502843,339513.063623
3,5445338,-169683.872845,0.026503,1688.348826,339381.745689,339463.306469
4,5492054,-170109.357061,0.022142,1262.864609,340232.714122,340314.274902
...,...,...,...,...,...,...
228,7915350,-169668.342411,0.024588,1703.879259,339350.684823,339432.245603
229,7920299,-169736.009130,0.024383,1636.212540,339486.018260,339567.579040
230,7938712,-169828.285298,0.022940,1543.936372,339670.570596,339752.131376
231,8058703,-169712.025385,0.024454,1660.196285,339438.050770,339519.611550


In [ ]:
conn, c = create_connection_cursor(SQL_DB)
c.execute("""
WITH FilteredModel AS (
    SELECT StoryWordID, SurprisalScore
    FROM ModelSurprisalScores
    WHERE ModelID = 6486044
)
SELECT SPRTNaturalStories.RTUID, 
       SPRTNaturalStories.WorkerID, 
       SPRTNaturalStories.StoryWordID, 
       SPRTNaturalStories.RT, 
       WordDetails.Word AS WordCategory, 
       WordDetails.CharacterLength, 
       WordDetails.WordUID AS WordCategoryID,
       WordDetails.LogFrequencies AS LogFrequencies,
       Story.POSTag AS POSTag,
       FilteredModel.SurprisalScore AS SurprisalScore
FROM SPRTNaturalStories
INNER JOIN Story ON SPRTNaturalStories.StoryWordID = Story.StoryWordID 
INNER JOIN WordDetails ON WordDetails.WordUID = Story.WordUID 
INNER JOIN FilteredModel ON FilteredModel.StoryWordID = Story.StoryWordID;

          """)

for row in c.fetchall():
    print(row)

Exception ignored in: <bound method IPythonKernel._clean_thread_parent_frames of <ipykernel.ipkernel.IPythonKernel object at 0x781c838f2a90>>
Traceback (most recent call last):
  File "/home/abishekthamma/PycharmProjects/masters_thesis/mt1/lib/python3.9/site-packages/ipykernel/ipkernel.py", line 770, in _clean_thread_parent_frames
    def _clean_thread_parent_frames(
KeyboardInterrupt: 


In [11]:

# List down each tokenizers and for each tokenizer, write it into the Tokenizer Table first and then use the tokenizer to tokenize the story and write it into the TokenizedStory Table. More easily, write it as a function that takes as input tokenizer and returns a tokenized story. 

#Alternatively, read the rows from the Story Table and for each row, tokenize it and write it into the TokenizedStory Table (Is this more optimal?) 


conn, c = create_connection_cursor(SQL_DB)

data_dir = "babylm_full_bpe_8k"
tokenizer = load_tokenizer(os.path.join(TOKENIZER_ROOT, data_dir))

#Insert into the Tokenizer Table if not already present
c.execute("SELECT TokenizerID FROM Tokenizer WHERE TokenizerName=?", (data_dir,))
if c.fetchone() is None:
    c.execute("INSERT INTO Tokenizer (TokenizerName, VocabularySize) VALUES (?, ?)", (data_dir, tokenizer.vocab_size))
else:
    print("Tokenizer already present in the database")
conn.commit()
conn.close()


Loading custom tokenizer from /home/abishekthamma/PycharmProjects/masters_thesis/ss-llm/nanoGPT/data/babylm_full_bpe_8k
Tokenizer already present in the database


In [52]:
def write_tokenizer_to_db(data_dir, tokenizer):
    conn, c = create_connection_cursor(SQL_DB)
    tokenizer = load_tokenizer(os.path.join(TOKENIZER_ROOT, data_dir))

    #Insert into the Tokenizer Table if not already present
    c.execute("SELECT TokenizerID FROM Tokenizer WHERE TokenizerName=?", (data_dir,))
    if c.fetchone() is None:
        c.execute("INSERT INTO Tokenizer (TokenizerName, VocabularySize) VALUES (?, ?)", (data_dir, tokenizer.vocab_size))
    else:
        print("Tokenizer already present in the database")
    conn.commit()
    conn.close()
    
def write_tokenized_story_to_db(tokenizer_name):
    """
    Given a tokenizer, this function encodes all stories in the "Story" Table in the database and writes it into TokenizedStory Table
    
    Selects all stories from story table and concatenates all words in the story. Passes this into the tokenizer story by story and writes the tokenized story into the TokenizedStory Table
    
    Intermediately, has to keep track of word vs token mapping as one word can have multiple tokens. Writes it into the TokenizedStory Table as 1 row for each token (Possibly multiple rows for each word)
        
    :param tokenizer_name: 
    :param  
    :return: 
    """
    
    conn, c = create_connection_cursor(SQL_DB)
    c.execute('SELECT CorpusID, StoryID, group_concat(Word," ") as FullStory FROM Story GROUP BY CorpusID, StoryID')
    stories = c.fetchall()
    
    c.execute("SELECT TokenizerID FROM Tokenizer WHERE TokenizerName=?", (tokenizer_name,))
    tokenizer_id = c.fetchone()[0]

    for story in stories:
        corpus_id = story[0]
        story_id = story[1]
        story_text = story[2]
        tokenized_story = tokenizer.encode(story_text)
        
        c.execute("SELECT StoryWordID, Word FROM Story WHERE CorpusID=? AND StoryID=? ORDER BY WordID", (corpus_id, story_id))
        words = c.fetchall()
        
        token_index = 0        
        for i, word in tqdm.tqdm(enumerate(words)):
            storyword_id = word[0]
            decode_list = []
            while True:
                if token_index >= len(tokenized_story):
                    break
                decode_list.append(tokenized_story[token_index])
                token_index += 1
                if tokenizer.decode(decode_list).strip() == word[1].lower():
                    break
            for j, token in enumerate(decode_list):
                c.execute("INSERT INTO TokenizedStory (TokenizerID, StoryWordID, TokenKey, TokenValue) VALUES (?, ?, ?, ?)", (tokenizer_id, storyword_id, j+1, token))
                
                if c.rowcount == 0:
                    print(f"Error writing tokenized story for {storyword_id}")
                    break
    conn.commit()
    conn.close()

tokenizer_list = ["babylm_full_bpe_8k", "babylm_full_bpe", "babylm_full_bpe_100M_8k", "babylm_wocdes_full_bpe"]

for tokenizer_names in tokenizer_list:
    tokenizer = load_tokenizer(os.path.join(TOKENIZER_ROOT, tokenizer_names))
    write_tokenizer_to_db(tokenizer_names, tokenizer)
    write_tokenized_story_to_db(tokenizer_names)
                

Loading custom tokenizer from /home/abishekthamma/PycharmProjects/masters_thesis/ss-llm/nanoGPT/data/babylm_full_bpe_8k
Loading custom tokenizer from /home/abishekthamma/PycharmProjects/masters_thesis/ss-llm/nanoGPT/data/babylm_full_bpe_8k
Tokenizer already present in the database


0it [00:00, ?it/s]


IntegrityError: UNIQUE constraint failed: TokenizedStory.TokenizerID, TokenizedStory.StoryWordID, TokenizedStory.TokenKey

In [124]:
#FUNCTION TO READ A STORY FROM TOKENIZED TABLE AND PASS IT TO MODEL TO GET SURPRISAL SCORES

def get_surprisal_inputs(model_id, story_id):
    conn, c = create_connection_cursor(SQL_DB)

    #For a given model, get its tokenizer id and context window size
    c.execute("SELECT TokenizerID, BlockSize FROM Model WHERE ModelID=?", (model_id,))
    model_row = c.fetchone()
    if model_row is None:
        print("Model ID not found in the database")
        return None
    
    tokenizer_id = model_row[0]
    context_window = model_row[1]
    
    #Query to get the context window of n or less words before the word given a story id
    query_fin = f"""  WITH    StoryWordRank 
                        AS  (SELECT  Story.StoryWordID,  
                                    Story.StoryID, 
                                    TokenizedStory.TokenValue,
                                    row_number() 
                                        OVER (
                                            PARTITION BY Story.CorpusID, Story.StoryID) AS 
                                    story_token_rank
                            FROM    TokenizedStory
                                    JOIN Story 
                                        On TokenizedStory.StoryWordID = Story.StoryWordID
                            WHERE TokenizedStory.TokenizerID = {tokenizer_id} AND Story.StoryID = {story_id})
                    SELECT  StoryWordRank.StoryWordID,
                            StoryWordRank.StoryID, 
                            StoryWordRank.TokenValue,
                            group_concat(TokenValue) 
                                OVER (
                                    PARTITION BY StoryID 
                                    ORDER BY story_token_rank ROWS BETWEEN {context_window} PRECEDING AND 1 PRECEDING)  
                                    
                                As 
                                ContextWindow 
                    FROM StoryWordRank """
    
    c.execute(query_fin)
    #Query returns StoryID, Word, ContextWindow
    tokenwise_context_story = c.fetchall()
 
    conn.close()
    return tokenwise_context_story

def batch_input_tokens(token_list, context_window):
    """
    Given a list of tokens, batch them into a tensor of size (batch_size, context_window)
    :param token_list: 
    :param batch_size: 
    :param context_window: 
    :return: 
    """
    batch_size = len(token_list) #since it could be less than the batch size in the last batch
    input_tensor = torch.zeros(batch_size, context_window, dtype=torch.int64)
    
    output_tensor = torch.zeros(batch_size, 1, dtype=torch.int64)
    
    for i, row in enumerate(token_list):
        if row[3] is not None:
            context_row = [int(token) for token in row[3].split(",")]
            input_tensor[i, -len(context_row):] = torch.tensor(context_row, dtype=torch.int64)
        else:
            input_tensor[i, -1] = 0
        output_tensor[i, 0] = int(row[2])
    
    input_tensor = input_tensor
    return input_tensor, output_tensor



#conn, c = create_connection_cursor(SQL_DB)

batch_size = 32
tokenwise_context_story = get_surprisal_inputs(model_id, 1)
print(tokenwise_context_story[:10])

for i in tqdm.tqdm(range(0, len(tokenwise_context_story), batch_size)):
    batch = tokenwise_context_story[i:i+batch_size]
    input_tensor, output_tensor = batch_input_tokens(batch, 256)
    surprisal_scores = return_surprisals(model, input_tensor, output_tensor) #Shape is (batch_size, )
    print(surprisal_scores)
    break
    

[(1, 1, '340', None), (2, 1, '215', '340'), (3, 1, '415', '340,215'), (4, 1, '205', '340,215,415'), (5, 1, '4771', '340,215,415,205'), (6, 1, '205', '340,215,415,205,4771'), (7, 1, '182', '340,215,415,205,4771,205'), (8, 1, '1239', '340,215,415,205,4771,205,182'), (9, 1, '211', '340,215,415,205,4771,205,182,1239'), (10, 1, '2369', '340,215,415,205,4771,205,182,1239,211')]


/tmp/ipykernel_113200/2485545820.py:120: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  output_tensor = torch.tensor(output_tensor).to(device)


tensor([10.0247,  6.4792,  6.2703,  5.9128,  9.9598,  5.0248,  2.0559,  7.9322,
         2.6659,  4.9624,  2.8235,  4.3886,  2.8920,  7.8896,  1.5405,  3.0038,
         9.5076,  3.8731,  3.4502,  8.4574,  0.4360, 10.1724,  4.8048,  4.6154,
         6.8957,  0.1090, 10.4410,  2.6252,  6.2390,  1.6029,  5.0428,  3.0057],
       device='cuda:0')


In [41]:
# tokenizer = load_tokenizer(os.path.join(TOKENIZER_ROOT, "babylm_full_bpe_8k"))
# print(tokenizer.pre_tokenizer)
# from tokenizers import pre_tokenizers
# tokenizer.pre_tokenizer = pre_tokenizers.ByteLevel()

print(tokenizer.tokenize("Hello World"))
print(tokenizer.encode("Hello World", return_tensors="pt"))

print(tokenizer.tokenize("\tJourney"))
print(tokenizer.encode("ĠJourney", return_tensors="pt"))

['hello', 'Ġworld']
tensor([[1893,  771]])
['j', 'our', 'ney']
tensor([[  45,   48,  321, 1522]])


In [10]:

#Insert Model data from the excel file into the Model Table

model_data_path = "/home/abishekthamma/PycharmProjects/masters_thesis/ss-llm/nanoGPT/results/rundata.xlsx"
model_data_df = pd.read_excel(model_data_path, sheet_name="Run Details")
model_data_df["tokenizer"] = model_data_df["dataset"]

model_data_df.head()

model_columns_to_db_columns = {
    "run_id": "ModelID",
    "output_folder_name": "OutputFolderName",
    "tokenizer": "Tokenizer", #Need to convert this to TokenizerID from Tokenizer Table
    "n_layer": "NumLayers",
    "n_head": "NumHeads",
    "block_size": "BlockSize",
    "n_embd": "EmbeddingDimension",
    "batch_size": "BatchSize",
    "learning_rate": "LearningRate",
    "seed": "Seed",
    "masking": "Masking",
    "mask_type": "MaskType",
    "mask_decay_rate": "MaskDecayRate",
    "echoic_memory": "EchoicMemory",
    "curriculum_learning": "CurriculumLearning",
    "curriculum_type": "CurriculumType",
    "dataset": "Dataset"
}

model_df_db = model_data_df[model_columns_to_db_columns.keys()].rename(columns=model_columns_to_db_columns)
model_df_db["Masking"] = model_df_db["Masking"].astype(bool)
model_df_db["CurriculumLearning"] = model_df_db["CurriculumLearning"].astype(bool)
model_df_db["MaskDecayRate"] = model_df_db["MaskDecayRate"].astype(float)
model_df_db["LearningRate"] = model_df_db["LearningRate"].astype(float)

model_df_db.head()
    
#Ensure not null constraints from create command are satisfied
# Not null constraints on following columns - ModelID, OutputFolderName, TokenizerID, NumLayers, NumHeads, BlockSize, EmbeddingDimension, BatchSize, LearningRate, Seed, Masking, MaskType, MaskDecayRate, EchoicMemory, CurriculumLearning, Dataset
constraint_check_columns = ["ModelID", "OutputFolderName", "Tokenizer", "NumLayers", "NumHeads", "BlockSize", "EmbeddingDimension", "BatchSize", "LearningRate", "Seed", "Masking", "MaskType", "MaskDecayRate", "EchoicMemory", "CurriculumLearning", "Dataset"]

row_ids = set()
for column in constraint_check_columns:
    if model_df_db[column].isnull().any():
        print(f"Checking for null values in {column}")
        print(model_df_db[model_df_db[column].isnull()].index)
        row_ids.update(model_df_db[model_df_db[column].isnull()].index)

#Drop rows with null values
model_df_db.drop(row_ids, inplace=True)

#For unique IDS in Tokenizer column, get the TokenizerID from the Tokenizer Table
conn, c = create_connection_cursor(SQL_DB)
tokenizer_ids = model_df_db["Tokenizer"].unique()
c.execute("SELECT TokenizerName, TokenizerID FROM Tokenizer WHERE TokenizerName IN ({})".format(','.join('?' * len(tokenizer_ids))), tokenizer_ids)
tokenizer_id_mapping = c.fetchall()
tokenizer_id_mapping = dict(tokenizer_id_mapping)

model_df_db["TokenizerID"] = model_df_db["Tokenizer"].map(tokenizer_id_mapping)

#Drop the Tokenizer column
model_df_db.drop(columns=["Tokenizer"], inplace=True)
model_df_db.head()

#Get model ids that are already present in the database
existing_model_ids = pd.read_sql("SELECT DISTINCT ModelID FROM Model", conn)["ModelID"].unique().tolist()

#Filter out models that are already present in the database
model_df_db = model_df_db[~model_df_db["ModelID"].isin(existing_model_ids)]

#Insert into the Model Table
model_df_db.to_sql("Model", conn, if_exists="append", index=False)

conn.close()

Checking for null values in Tokenizer
Index([26], dtype='int64')
Checking for null values in BatchSize
Index([26], dtype='int64')
Checking for null values in LearningRate
Index([26], dtype='int64')
Checking for null values in Dataset
Index([26], dtype='int64')
